## Split Overmerged Authors — Phase 2: Different First Name, Same Last Name

Identifies work_authors where the parsed **last name** of `raw_author_name` matches the author profile's parsed last name, but the **first names** are clearly different.

**Safety filters:**
- Requires both first/last names > 2 chars
- No substring overlap on first name in either direction (raw or hyphen-stripped)
- Excludes CJK-source `raw_author_name` (Hangul/Hiragana/Katakana/Han) — high romanization FP rate
- Excludes CJK-source profile `full_name`
- Excludes periods on either parsed first name (initials like "Th.")
- Excludes title/degree tokens as first name (md, lord, sc., ltcol, &na, etc.)
- Excludes profiles with ORCID (conservative — ORCID anchor is strong)

**Design decision — no co-author overlap filter.** A co-author overlap filter was evaluated and dropped. It would have cut the FP rate from ~11% to ~7%, but at the cost of missing ~270 true overmerges for every ~58 false positives avoided (~4.7x recall cost). The Phase 1 playbook (ship aggressive, rollback discovered systematic FP classes via the audit log) is preferred instead — see `oxjobs/working/split-authors-name-parser/eval/findings.md`.

Evaluation on 100 stratified authors (1,268 candidate work_authors) using the `aer-gold-standard` Sonnet judge: **~10.7% per-candidate FP rate**. Projected production scale: ~3.3M splits, ~350K expected wrong splits — manageable via rollback audit log.

### Cell 1: Build candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_candidates_phase2 AS
WITH work_parsed AS (
  SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name,
    an_w.parsed_name.last AS work_last,
    an_w.parsed_name.first AS work_first
  FROM openalex.works.work_authors wa
  JOIN openalex.authors.author_names an_w ON wa.raw_author_name = an_w.raw_author_name
  WHERE wa.author_id IS NOT NULL
    AND an_w.parsed_name.last IS NOT NULL
    AND LENGTH(an_w.parsed_name.last) > 2
    AND LENGTH(an_w.parsed_name.first) > 2
    AND NOT REGEXP_LIKE(wa.raw_author_name, '[\uac00-\ud7af\u3040-\u30ff\u4e00-\u9fff]')
),
author_parsed AS (
  SELECT oa.id AS author_id,
    oa.full_name AS author_full_name,
    an_a.parsed_name.last AS author_last,
    an_a.parsed_name.first AS author_first,
    oa.orcid, oa.works_count
  FROM openalex.authors.openalex_authors oa
  JOIN openalex.authors.author_names an_a ON oa.full_name = an_a.raw_author_name
  WHERE an_a.parsed_name.last IS NOT NULL
    AND LENGTH(an_a.parsed_name.last) > 2
    AND LENGTH(an_a.parsed_name.first) > 2
    AND NOT REGEXP_LIKE(oa.full_name, '[\uac00-\ud7af\u3040-\u30ff\u4e00-\u9fff]')
)
SELECT wp.work_id, wp.author_sequence, wp.author_id, wp.raw_author_name,
  wp.work_last, wp.work_first, ap.author_last, ap.author_first,
  ap.author_full_name, ap.orcid, ap.works_count
FROM work_parsed wp
JOIN author_parsed ap ON wp.author_id = ap.author_id
WHERE wp.work_last = ap.author_last
  AND wp.work_first != ap.author_first
  AND INSTR(wp.work_first, ap.author_first) = 0
  AND INSTR(ap.author_first, wp.work_first) = 0
  AND INSTR(TRANSLATE(wp.work_first, '-', ''), TRANSLATE(ap.author_first, '-', '')) = 0
  AND INSTR(TRANSLATE(ap.author_first, '-', ''), TRANSLATE(wp.work_first, '-', '')) = 0
  AND wp.work_first NOT LIKE '%.%'
  AND ap.author_first NOT LIKE '%.%'
  AND wp.work_first NOT IN ('md','mohamed','mohammed','muhammad','lord','lady','sir','dame','ltcol','capt','col','rev','hon','prof','doctor')
  AND ap.author_first NOT IN ('md','mohamed','mohammed','muhammad','lord','lady','sir','dame','ltcol','capt','col','rev','hon','prof','doctor','&na')
  AND ap.orcid IS NULL

### Cell 2: Validation statistics

In [ ]:
SELECT COUNT(*) AS total_candidates,
  COUNT(DISTINCT author_id) AS distinct_authors,
  COUNT(DISTINCT work_id) AS distinct_works,
  PERCENTILE_APPROX(works_count, ARRAY(0.25, 0.5, 0.75, 0.95)) AS author_works_pctiles
FROM openalex.authors.overmerge_split_candidates_phase2

### Cell 3: Spot-check sample (manual review)

In [ ]:
SELECT work_id, author_id, raw_author_name, author_full_name,
  work_first, author_first, work_last,
  works_count AS author_works_count
FROM openalex.authors.overmerge_split_candidates_phase2
ORDER BY RAND()
LIMIT 50

### Cell 4: Create audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_log_phase2 AS
SELECT work_id, author_sequence, author_id AS previous_author_id,
  raw_author_name, work_first, author_first, work_last,
  current_timestamp() AS split_timestamp
FROM openalex.authors.overmerge_split_candidates_phase2

### Cell 5: Execute the split

**WARNING**: This nulls out author_ids. Verify cells 2-3 before running.

**NOTE**: MatchAuthors has cutoffs (`work_id > 7B`, `created_date >= 2025-12-20`). Older split records will NOT be re-processed automatically. A separate re-matching run or cutoff relaxation is needed (same issue as Phase 1).

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT DISTINCT work_id, author_sequence
  FROM openalex.authors.overmerge_split_candidates_phase2
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 6: Queue split works for reprocessing by UpdateWorkAuthorships

Push affected work_ids into `curated_work_ids_pending_sync` so the next `UpdateWorkAuthorships` run picks them up and propagates the nulled author_ids through to `work_authorships` → `openalex_works`.

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_phase2

### Cell 7: Post-split verification

In [ ]:
SELECT
  COUNT(*) AS nulled_records,
  COUNT(DISTINCT c.work_id) AS works_affected
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase2 c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE wa.author_id IS NULL

### Cell 8: Outcome tracking (run after MatchAuthors re-processes)

In [ ]:
SELECT
  CASE
    WHEN wa.author_id IS NULL THEN 'STILL_UNMATCHED'
    WHEN wa.author_id = log.previous_author_id THEN 'RE_MATCHED_SAME'
    ELSE 'RE_MATCHED_NEW'
  END AS outcome,
  COUNT(*) AS cnt
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_phase2 log
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
GROUP BY 1

### Cell 9: Rollback template (use if systematic FP class discovered)

Phase 2's predicted FP rate is ~11%. If a systematic FP class is discovered post-split (e.g., a specific parser quirk or name-change pattern — similar to Phase 1's Kyle Demes case), rollback uses the same pattern as Phase 1:

```sql
MERGE INTO openalex.works.work_authors AS target
USING openalex.authors.overmerge_split_log_phase2 AS source
ON target.work_id = source.work_id AND target.author_sequence = source.author_sequence
WHEN MATCHED AND target.author_id IS NULL THEN
  UPDATE SET target.author_id = source.previous_author_id
```

Add a `WHERE` clause inside the USING subquery to restrict the rollback to the discovered FP class. Don't run this cell as-is — it would restore all splits.